# 2D Infection Spread

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ryouy/simulations/blob/main/simulations/infection-spread/infection_simulation.ipynb)

最初のセルで関数を読み込み、最後のセルで設定して実行します。表示上の球の半径と衝突判定の半径を一致させ、移動を内部サブステップに分割して接触の見逃しを減らしています。

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from scipy.spatial import cKDTree
from IPython.display import display, Image, clear_output
from pathlib import Path
from tqdm.auto import tqdm
import ipywidgets as widgets
import csv, json, time, math

OUT = Path("outputs")
OUT.mkdir(exist_ok=True)
REPO = "ryouy/simulations"
PREFIX = "simulations/infection-spread"

def simulate(n, radius, speed, steps, dt, turn_noise, seed):
    rng = np.random.default_rng(seed)
    if n < 2 or radius <= 0 or speed <= 0 or dt <= 0:
        raise ValueError("球の数・半径・速度・時間刻みを確認してください")
    # Batch placement, rejecting only overlapping points.
    pos = rng.uniform(radius, 1-radius, size=(n, 2))
    for _ in range(200):
        pairs = cKDTree(pos).query_pairs(2.05*radius, output_type="ndarray")
        if not len(pairs):
            break
        bad = np.unique(pairs[:,1])
        pos[bad] = rng.uniform(radius, 1-radius, size=(len(bad),2))
    else:
        raise ValueError("初期配置が密集しすぎています。球の数または半径を小さくしてください。")
    angles = rng.uniform(0, 2*np.pi, n)
    vel = speed*np.column_stack((np.cos(angles), np.sin(angles)))
    infected = np.zeros(n,dtype=bool)
    infected[0] = True
    frames = [pos.copy()]
    states = [infected.copy()]
    counts = [1]
    # Substeps prevent missed crossings; keep displacement <= radius/2.
    substeps = max(1, math.ceil(speed*dt/(radius*0.5)))
    if steps*substeps > 1_000_000:
        raise ValueError("計算量が大きすぎます。半径を大きくするか、速度・ステップ数を減らしてください。")
    print(f"計算開始: {n}個 / {steps}ステップ / 内部サブステップ {substeps}回")
    with tqdm(total=steps,desc="Simulation",unit="step") as progress:
        for step in range(steps):
            angles = np.arctan2(vel[:,1],vel[:,0]) + rng.normal(0,turn_noise*np.sqrt(dt),n)
            vel = speed*np.column_stack((np.cos(angles),np.sin(angles)))
            h = dt/substeps
            for _ in range(substeps):
                pos += vel*h
                for axis in (0,1):
                    low = pos[:,axis] < radius
                    high = pos[:,axis] > 1-radius
                    pos[low,axis] = 2*radius-pos[low,axis]
                    pos[high,axis] = 2*(1-radius)-pos[high,axis]
                    vel[low|high,axis] *= -1
                pairs = cKDTree(pos).query_pairs(2*radius,output_type="ndarray")
                for i,j in pairs:
                    if infected[i] or infected[j]:
                        infected[i] = infected[j] = True
                    delta = pos[j]-pos[i]
                    dist = np.linalg.norm(delta)
                    normal = delta/dist if dist > 1e-12 else np.array([1.,0.])
                    overlap = max(0.,2*radius-dist)
                    pos[i] -= normal*overlap/2
                    pos[j] += normal*overlap/2
                    relative = np.dot(vel[i]-vel[j],normal)
                    if relative > 0:
                        vel[i] -= relative*normal
                        vel[j] += relative*normal
                np.clip(pos,radius,1-radius,out=pos)
            frames.append(pos.copy())
            states.append(infected.copy())
            counts.append(int(infected.sum()))
            progress.update(1)
            if step % 10 == 0:
                progress.set_postfix(infected=f"{counts[-1]}/{n}")
    return frames, states, counts, substeps

def export_results(frames,states,counts,config):
    n=config["N"]
    with (OUT/"infection_counts.csv").open("w",newline="") as f:
        writer=csv.writer(f)
        writer.writerow(["step","time","infected"])
        writer.writerows((i,i*config["DT"],v) for i,v in enumerate(counts))
    (OUT/"settings.json").write_text(json.dumps(config,indent=2,ensure_ascii=False))
    fig,ax=plt.subplots(figsize=(6,3.2))
    ax.plot(np.arange(len(counts))*config["DT"],counts,color="#d84343")
    ax.set(xlabel="Time",ylabel="Infected",ylim=(0,n+1),title="Infection growth")
    fig.tight_layout()
    fig.savefig(OUT/"infection_curve.png",dpi=150)
    plt.close(fig)

    fig,ax=plt.subplots(figsize=(6,6),facecolor="#111111")
    ax.set_facecolor("#111111")
    ax.set_xlim(0,1); ax.set_ylim(0,1); ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values(): spine.set_color("#777777")
    # Scatter markers sized from physical radius, using axes width in points.
    # Avoid PatchCollection.set_paths entirely (caused Path/get_transform errors).
    fig.canvas.draw()
    axes_width_points = ax.get_window_extent().width * 72 / fig.dpi
    diameter_points = 2 * config["RADIUS"] * axes_width_points
    marker_area = np.pi * (diameter_points / 2) ** 2
    scatter = ax.scatter([], [], s=marker_area, linewidths=0,
                         c="#ffffff", edgecolors="none")
    label=ax.text(.02,1.02,"",transform=ax.transAxes,color="white")
    skip=max(1,math.ceil(len(frames)/180))
    indices=list(range(0,len(frames),skip))
    if indices[-1] != len(frames)-1: indices.append(len(frames)-1)
    def draw(k):
        scatter.set_offsets(frames[k])
        scatter.set_facecolors(np.where(states[k],"#e53935","#ffffff"))
        label.set_text(f"Step {k} | Infected {counts[k]}/{n}")
        return scatter,label
    animation=FuncAnimation(fig,draw,frames=indices,interval=1000/15,blit=False,repeat=False)
    gif=OUT/"infection.gif"
    with tqdm(total=len(indices),desc="Rendering GIF",unit="frame") as progress:
        class ProgressWriter(PillowWriter):
            def grab_frame(self,**kwargs):
                super().grab_frame(**kwargs)
                progress.update(1)
        animation.save(str(gif),writer=ProgressWriter(fps=15),dpi=95)
    plt.close(fig)
    print(f"GIF生成完了: {gif} ({gif.stat().st_size/1024/1024:.1f} MB)")
    display(Image(filename=str(gif)))


## 操作パネル

下のセルを実行するとスライダーと実行ボタンが表示されます。**GitHubアップロード**を選択すると、生成したGIF・グラフ・CSV・設定をリポジトリに送信します。各シミュレーションのREADMEは `outputs/infection.gif` を参照するため、アップロード後に自動で最新GIFが表示されます。

注：内部サブステップは離散的な近似であり、厳密な連続衝突検出ではありません。球を小さくすると計算負荷が増えます。

In [4]:
# 最後のセル：設定 → 計算 → GIF → GitHub保存
n_widget=widgets.IntSlider(value=80,min=10,max=1000,step=10,description="球の数",continuous_update=False,style={"description_width":"130px"})
r_widget=widgets.FloatLogSlider(value=0.005,base=10,min=-3,max=-1.7,step=.05,description="球の半径",readout_format=".4f",continuous_update=False,style={"description_width":"130px"})
s_widget=widgets.FloatSlider(value=.22,min=.02,max=.5,step=.01,description="速度",continuous_update=False,style={"description_width":"130px"})
steps_widget=widgets.IntSlider(value=600,min=100,max=3200,step=100,description="ステップ数",continuous_update=False,style={"description_width":"130px"})
dt_widget=widgets.FloatSlider(value=.025,min=.005,max=.05,step=.005,description="時間刻み",readout_format=".3f",continuous_update=False,style={"description_width":"130px"})
noise_widget=widgets.FloatSlider(value=.35,min=0,max=2,step=.05,description="方向の揺らぎ",continuous_update=False,style={"description_width":"130px"})
seed_widget=widgets.IntText(value=42,description="乱数シード",style={"description_width":"130px"})
upload_widget=widgets.Checkbox(value=False,description="生成後にGitHubへアップロード（Colab SecretsのGITHUB_TOKENが必要）",indent=False)
hint=widgets.HTML()
preview=widgets.Output()
run_button=widgets.Button(description="シミュレーションを実行してGIFを生成",button_style="primary",icon="play")
result=widgets.Output()
def update_hint(*_):
    n=n_widget.value;r=r_widget.value;speed=s_widget.value;dt=dt_widget.value
    sub=math.ceil(speed*dt/(r*.5))
    # Matplotlib preview works in Colab even when SVG in widgets.HTML is filtered.
    rng_preview=np.random.default_rng(123)
    coords=rng_preview.uniform(r,1-r,size=(min(n,350),2))
    with preview:
        clear_output(wait=True)
        fig_preview,ax_preview=plt.subplots(figsize=(4.5,4.5),facecolor="#111111")
        ax_preview.set_facecolor("#111111")
        ax_preview.set(xlim=(0,1),ylim=(0,1))
        ax_preview.set_aspect("equal")
        ax_preview.set_xticks([]);ax_preview.set_yticks([])
        for spine in ax_preview.spines.values(): spine.set_color("#888888")
        fig_preview.canvas.draw()
        width_pt=ax_preview.get_window_extent().width*72/fig_preview.dpi
        diameter_pt=2*r*width_pt
        area=np.pi*(diameter_pt/2)**2
        colors=np.array(["#b5c0cc"]*len(coords))
        colors[0]="#e53935"
        ax_preview.scatter(coords[:,0],coords[:,1],s=area,c=colors,linewidths=0)
        ax_preview.set_title("1 × 1 / actual scale",color="white",pad=10)
        display(fig_preview)
        plt.close(fig_preview)
        print(f"プレビュー: {len(coords)}個表示 / 設定 {n}個（最大350個を表示）")
    hint.value=(f"<b>球の直径:</b> {2*r:.4f}　<b>1ステップの移動:</b> {speed*dt:.4f}　"
                f"<b>内部サブステップ:</b> {sub}　<b>推定内部更新回数:</b> {steps_widget.value*sub:,}<br>"
                "<small>表示する球の半径は衝突判定の半径と同じです。小さな球は画面でも小さく表示されます。</small>")
for w in (n_widget,r_widget,s_widget,steps_widget,dt_widget):
    w.observe(update_hint,names="value")
update_hint()
def upload_outputs():
    from google.colab import userdata
    import requests,base64
    token=userdata.get("GITHUB_TOKEN")
    if not token: raise ValueError("Colab SecretsにGITHUB_TOKENを設定してください")
    headers={"Authorization":f"Bearer {token}","Accept":"application/vnd.github+json","X-GitHub-Api-Version":"2022-11-28"}
    for file in sorted(OUT.iterdir()):
        if not file.is_file() or file.suffix.lower() not in (".gif",".png",".csv",".json"): continue
        remote=f"{PREFIX}/outputs/{file.name}"
        url=f"https://api.github.com/repos/{REPO}/contents/{remote}"
        existing=requests.get(url,headers=headers,timeout=30)
        if existing.status_code not in (200,404): existing.raise_for_status()
        payload={"message":f"Update infection simulation: {file.name}","content":base64.b64encode(file.read_bytes()).decode(),"branch":"main"}
        if existing.status_code==200: payload["sha"]=existing.json()["sha"]
        response=requests.put(url,headers=headers,json=payload,timeout=120)
        response.raise_for_status()
        print("Uploaded:",remote)
    print("READMEのGIFプレビューは同じファイルパスから自動更新されます。")
def on_run(_):
    run_button.disabled=True
    with result:
        clear_output(wait=True)
        try:
            config={"N":n_widget.value,"RADIUS":r_widget.value,"SPEED":s_widget.value,
                    "STEPS":steps_widget.value,"DT":dt_widget.value,
                    "TURN_NOISE":noise_widget.value,"SEED":seed_widget.value}
            start=time.monotonic()
            frames,states,counts,substeps=simulate(
                config["N"],config["RADIUS"],config["SPEED"],config["STEPS"],
                config["DT"],config["TURN_NOISE"],config["SEED"])
            print(f"計算終了: 感染 {counts[-1]}/{config['N']} ({100*counts[-1]/config['N']:.1f}%)")
            export_results(frames,states,counts,config)
            print(f"合計処理時間: {time.monotonic()-start:.1f}秒")
            if upload_widget.value: upload_outputs()
            else: print("GitHubアップロードは未選択です。READMEへの反映にはアップロードを有効にして再実行してください。")
        except Exception as e:
            print(f"処理を完了できませんでした: {type(e).__name__}: {e}")
            raise
        finally:
            run_button.disabled=False
run_button.on_click(on_run)
display(widgets.VBox([
    widgets.HTML("<h3>Infection Spread — Simulation Controls</h3>"),
    n_widget,r_widget,s_widget,steps_widget,dt_widget,noise_widget,seed_widget,
    hint,preview,upload_widget,run_button,result
]))
